<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Visualization**


Estimated time needed: **45** minutes


In this lab, you will focus on data visualization. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


After completing this lab, you will be able to:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize composition and comparison of data.




## Demo: How to work with database


Download the database file.


In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

**Install and Import Necessary Python Libraries**

Ensure that you have the required libraries installed to work with SQLite and Pandas:


In [ ]:
!pip install pandas 
!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt

**Read the CSV File into a Pandas DataFrame**

Load the Stack Overflow survey data into a Pandas DataFrame:


In [ ]:
# Read the CSV file
df = pd.read_csv('survey-data.csv')

# Display the first few rows of the data
df.head()


**Create a SQLite Database and Insert the Data**

Now, let's create a new SQLite database (`survey-data.sqlite`) and insert the data from the DataFrame into a table using the sqlite3 library:


In [ ]:
import sqlite3

# Create a connection to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Write the dataframe to the SQLite database
df.to_sql('main', conn, if_exists='replace', index=False)


# Close the connection
conn.close()


**Verify the Data in the SQLite Database**
Verify that the data has been correctly inserted into the SQLite database by running a simple query:


In [ ]:
# Reconnect to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Run a simple query to check the data
QUERY = "SELECT * FROM main LIMIT 5"
df_check = pd.read_sql_query(QUERY, conn)

# Display the results
print(df_check)


## Demo: Running an SQL Query


Count the number of rows in the table named 'main'


In [ ]:
QUERY = """
SELECT COUNT(*) 
FROM main
"""
df = pd.read_sql_query(QUERY, conn)
df.head()


## Demo: Listing All Tables


To view the names of all tables in the database:


In [ ]:
QUERY = """
SELECT name as Table_Name FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


## Demo: Running a Group By Query
    
For example, you can group data by a specific column, like Age, to get the count of respondents in each age group:


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count
FROM main
GROUP BY Age
ORDER BY Age
"""
pd.read_sql_query(QUERY, conn)


## Demo: Describing a table

Use this query to get the schema of a specific table, main in this case:


In [ ]:
table_name = 'main'

QUERY = """
SELECT sql FROM sqlite_master 
WHERE name= '{}'
""".format(table_name)

df = pd.read_sql_query(QUERY, conn)
print(df.iat[0,0])


## Hands-on Lab


### Visualizing the Distribution of Data

**Histograms**

Plot a histogram of CompTotal (Total Compensation).


In [ ]:
## Write your code here
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="CompTotal",
    bins=30,
    color="steelblue",
    edgecolor="black"
)

plt.title("Distribution of Total Compensation (CompTotal)", fontsize=14)
plt.xlabel("Total Compensation")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

**Box Plots**

Plot a box plot of Age.


In [ ]:
## Write your code here

plt.figure(figsize=(10, 4))

sns.boxplot(
    data=df,
    x="Age",
    color="steelblue"
)

plt.title("Distribution of Age")
plt.xlabel("Age")
plt.tight_layout()

plt.show()

### Visualizing Relationships in Data

**Scatter Plots**

Create a scatter plot of Age and WorkExp.


In [ ]:
## Write your code here

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Age",
    y="WorkExp",
    alpha=0.6
)

plt.title("Age vs Work Experience")
plt.xlabel("Age")
plt.ylabel("Work Experience (Years)")

plt.tight_layout

**Bubble Plots**

Create a bubble plot of `TimeSearching` and `Frustration` using the Age column as the bubble size.


In [13]:
## Write your code here

bubble_data = df[
    ["TimeSearching", "Frustration", "Age"]
].dropna().copy()

plt.figure(figsize=(12, 7))

sns.scatterplot(
    data=bubble_data,
    x="TimeSearching",
    y="Frustration",
    size="Age",
    sizes=(30, 300),
    alpha=0.5,
    legend="brief"
)

plt.title("Time Searching vs Frustration by Age")
plt.xlabel("Time Searching")
plt.ylabel("Frustration")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

KeyError: "None of [Index(['TimeSearching', 'Frustration', 'Age'], dtype='str')] are in the [columns]"

### Visualizing Composition of Data

**Pie Charts**

Create a pie chart of the top 5 databases(`DatabaseWantToWorkWith`) that respondents wish to learn next year.


In [14]:
## Write your code here
database_counts = (
    df["DatabaseWantToWorkWith"]
    .dropna()
    .str.split(";")
    .explode()
    .value_counts()
    .head(5)
)

plt.figure(figsize=(8, 8))

plt.pie(
    database_counts,
    labels=database_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Top 5 Databases Respondents Want to Work With Next Year")

plt.tight_layout()
plt.show()

KeyError: 'DatabaseWantToWorkWith'

**Stacked Charts** 

Create a stacked bar chart of median `TimeSearching` and `TimeAnswering` for the age group 30 to 35.


In [15]:
## Write your code here
time_mapping = {
    "Less than 15 minutes a day": 7.5,
    "15-30 minutes a day": 22.5,
    "30-60 minutes a day": 45,
    "60-120 minutes a day": 90,
    "Over 120 minutes a day": 120
}

age_30_35 = df.loc[
    df["Age"].between(30, 35),
    ["Age", "TimeSearching", "TimeAnswering"]
].copy()

age_30_35["TimeSearching_minutes"] = (
    age_30_35["TimeSearching"].map(time_mapping)
)

age_30_35["TimeAnswering_minutes"] = (
    age_30_35["TimeAnswering"].map(time_mapping)
)

median_time_by_age = (
    age_30_35
    .groupby("Age")[["TimeSearching_minutes", "TimeAnswering_minutes"]]
    .median()
)

median_time_by_age.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    edgecolor="black"
)

plt.title("Median Time Searching and Answering by Age")
plt.xlabel("Age")
plt.ylabel("Median Time per Day (Minutes)")
plt.xticks(rotation=0)
plt.legend(
    ["Time Searching", "Time Answering"],
    title="Activity"
)

plt.tight_layout()
plt.show()

KeyError: 'Age'

### Visualizing Comparison of Data

**Line Chart**

Plot the median `CompTotal` for all ages from 45 to 60.


In [16]:
## Write your code here
age_compensation = df[["Age", "CompTotal"]].copy()

age_compensation["Age"] = pd.to_numeric(
    age_compensation["Age"],
    errors="coerce"
)

age_compensation["CompTotal"] = pd.to_numeric(
    age_compensation["CompTotal"],
    errors="coerce"
)

median_comp_by_age = (
    age_compensation
    .loc[age_compensation["Age"].between(45, 60)]
    .dropna(subset=["Age", "CompTotal"])
    .groupby("Age", as_index=False)["CompTotal"]
    .median()
    .sort_values("Age")
)

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=median_comp_by_age,
    x="Age",
    y="CompTotal",
    marker="o"
)

plt.title("Median Total Compensation by Age (45–60)")
plt.xlabel("Age")
plt.ylabel("Median CompTotal")
plt.xticks(range(45, 61))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

KeyError: "None of [Index(['Age', 'CompTotal'], dtype='str')] are in the [columns]"

**Bar Chart**

Create a horizontal bar chart using the `MainBranch` column.


In [17]:
## Write your code here

mainbranch_counts = (
    df["MainBranch"]
    .value_counts()
    .sort_values()
)

plt.figure(figsize=(10, 6))

plt.barh(
    mainbranch_counts.index,
    mainbranch_counts.values
)

plt.title("Distribution of Respondents by Main Branch")
plt.xlabel("Number of Respondents")
plt.ylabel("Main Branch")

plt.tight_layout()
plt.show()

KeyError: 'MainBranch'

### Summary


In this lab, you focused on extracting and visualizing data from an RDBMS using SQL queries and SQLite. You applied various visualization techniques, including:

- Histograms to display the distribution of CompTotal.
- Box plots to show the spread of ages.
- Scatter plots and bubble plots to explore relationships between variables like Age, WorkExp, `TimeSearching` and `TimeAnswering`.
- Pie charts and stacked charts to visualize the composition of data.
- Line charts and bar charts to compare data across categories.


### Close the Database Connection

Once the lab is complete, ensure to close the database connection:


In [18]:
conn.close()

## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
